# ASH-KV Real-World Load & Quality Benchmark

This suite uses a genuine ShareGPT dataset and a Poisson-distributed asynchronous load generator to stress the LLM KV cache.
It measures the exact impact of INT8 HiCache compression on Time-To-First-Token (TTFT) and Time-Per-Output-Token (TPOT).

In [ ]:
!pip install aiohttp matplotlib requests numpy

## 1. Fetch & Prepare ShareGPT Dataset

In [ ]:
import requests
import json
import random
import os

DATASET_URL = "https://huggingface.co/datasets/anon8231489123/ShareGPT_Vicuna_unfiltered/resolve/main/ShareGPT_V3_unfiltered_cleaned_split.json"
DATASET_FILE = "sharegpt.json"

if not os.path.exists(DATASET_FILE):
    print("Downloading ShareGPT dataset...")
    r = requests.get(DATASET_URL)
    with open(DATASET_FILE, "w") as f:
        f.write(r.text)
    print("Download complete.")
else:
    print("ShareGPT dataset already exists.")

with open(DATASET_FILE, "r") as f:
    sharegpt_data = json.load(f)

# Extract human prompts
prompts = []
for conversation in sharegpt_data:
    for msg in conversation.get("conversations", []):
        if msg.get("from") == "human":
            if len(msg.get("value", "")) > 100:  # Filter out trivial prompts
                prompts.append(msg["value"])
                break
    if len(prompts) >= 500:
        break

print(f"Loaded {len(prompts)} real-world prompts.")

## 2. Load Generator Engine

In [ ]:
import asyncio
import aiohttp
import time
import numpy as np
import matplotlib.pyplot as plt

ENDPOINT = "http://localhost:30000/v1/chat/completions"
MODEL = "Qwen/Qwen2.5-7B-Instruct"

async def fetch(session, prompt):
    payload = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 100,  # Generate a fixed amount of tokens to measure TPOT
        "temperature": 0.0,
        "stream": True  # Use streaming to measure TTFT
    }
    
    start_time = time.time()
    ttft = None
    total_tokens = 0
    
    try:
        async with session.post(ENDPOINT, json=payload) as response:
            async for line in response.content:
                if line:
                    if ttft is None:
                        ttft = time.time() - start_time
                    total_tokens += 1
            end_time = time.time()
            
            tpot = (end_time - start_time - ttft) / max(1, total_tokens)
            return {"status": "success", "ttft": ttft, "tpot": tpot, "e2e": end_time - start_time}
    except Exception as e:
        return {"status": "failed", "error": str(e)}

async def run_benchmark(concurrency, num_requests, arrival_rate):
    print(f"Starting benchmark: {num_requests} requests at {arrival_rate} req/s...")
    results = []
    async with aiohttp.ClientSession() as session:
        tasks = []
        for i in range(num_requests):
            prompt = random.choice(prompts)
            task = asyncio.create_task(fetch(session, prompt))
            tasks.append(task)
            
            # Poisson arrival wait
            await asyncio.sleep(np.random.exponential(1.0 / arrival_rate))
            
        results = await asyncio.gather(*tasks)
    return results

def analyze_results(results, name):
    successes = [r for r in results if r["status"] == "success"]
    if not successes:
        print(f"{name} Benchmark FAILED completely.")
        return None
        
    ttfts = [r["ttft"] for r in successes]
    tpots = [r["tpot"] for r in successes]
    
    print(f"=== {name} RESULTS ===")
    print(f"Success Rate: {len(successes)}/{len(results)}")
    print(f"Avg TTFT: {sum(ttfts)/len(ttfts):.3f}s")
    print(f"Avg TPOT: {sum(tpots)/len(tpots):.3f}s")
    
    return {"ttfts": ttfts, "tpots": tpots, "success_rate": len(successes)/len(results)}


## 3. Baseline Execution (No ASH-KV)
**Start standard SGLang in your terminal:**
```bash
python -m sglang.launch_server --model-path Qwen/Qwen2.5-7B-Instruct --port 30000 --mem-fraction-static 0.3 --hicache-write-policy write_through
```

In [ ]:
baseline_results = await run_benchmark(concurrency=10, num_requests=30, arrival_rate=5)
baseline_stats = analyze_results(baseline_results, "BASELINE")

## 4. ASH-KV Execution
**Kill the baseline server and start ASH-KV patched SGLang:**
```bash
PYTHONPATH=. python tests/run_real_sglang_server.py --model Qwen/Qwen2.5-7B-Instruct --port 30000
```

In [ ]:
ashkv_results = await run_benchmark(concurrency=10, num_requests=30, arrival_rate=5)
ashkv_stats = analyze_results(ashkv_results, "ASH-KV")

## 5. Performance Comparison

In [ ]:
if baseline_stats and ashkv_stats:
    labels = ['Baseline', 'ASH-KV']
    ttft_means = [sum(baseline_stats['ttfts'])/len(baseline_stats['ttfts']), sum(ashkv_stats['ttfts'])/len(ashkv_stats['ttfts'])]
    tpot_means = [sum(baseline_stats['tpots'])/len(baseline_stats['tpots']), sum(ashkv_stats['tpots'])/len(ashkv_stats['tpots'])]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    
    ax1.bar(labels, ttft_means, color=['blue', 'green'])
    ax1.set_title('Avg TTFT (Time To First Token)')
    ax1.set_ylabel('Seconds')
    
    ax2.bar(labels, tpot_means, color=['blue', 'green'])
    ax2.set_title('Avg TPOT (Time Per Output Token)')
    ax2.set_ylabel('Seconds')
    
    plt.tight_layout()
    plt.show()
    
    ttft_degradation = (ttft_means[1] - ttft_means[0]) / ttft_means[0] * 100
    print(f"ASH-KV TTFT Overhead: {ttft_degradation:.2f}%")

## 6. Needle In A Haystack (Quality Validation)
This cell tests if the INT8 cache compression corrupts precise information retrieval.

In [ ]:
import requests
import time

haystack = "The Roman Empire was massive. " * 500
needle = "The secret password to the Roman vault is 'GLADIATOR99'. "
full_context = haystack + needle + haystack

print("Sending massive context block (Prefill)...")
payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": full_context + "\n\nWhat is the secret password to the vault?"}],
    "max_tokens": 50,
    "temperature": 0.0
}

r = requests.post(ENDPOINT, json=payload)
print("Response:", r.json()['choices'][0]['message']['content'])

print("\nNow, overflow the cache with other requests to force the NIAH prompt to compress into INT8...")
async def flood():
    async with aiohttp.ClientSession() as session:
        tasks = [fetch(session, random.choice(prompts)) for _ in range(15)]
        await asyncio.gather(*tasks)
        
await flood()

print("\nCache overflowed. Now asking a follow-up question on the exact same prefix (forcing decompression)...")
payload2 = {
    "model": MODEL,
    "messages": [
        {"role": "user", "content": full_context + "\n\nWhat is the secret password to the vault?"},
        {"role": "assistant", "content": "The secret password to the Roman vault is 'GLADIATOR99'."},
        {"role": "user", "content": "Can you repeat that password in uppercase?"}
    ],
    "max_tokens": 50,
    "temperature": 0.0
}

start = time.time()
r2 = requests.post(ENDPOINT, json=payload2)
print("Response:", r2.json()['choices'][0]['message']['content'])
print(f"Response Time (Decode hit): {time.time()-start:.2f}s")
if 'GLADIATOR99' in r2.json()['choices'][0]['message']['content'].upper():
    print("\n✅ NIAH PASSED: Semantic meaning perfectly preserved through INT8 compression/decompression!")
else:
    print("\n❌ NIAH FAILED: The model hallucinated or lost the context during compression.")